# SignLingo AI Mini — Training Notebook

**Objective:** build a small context-aware ASL pose **retrieval** system.

> Research prototype. This is retrieval (nearest dataset clip), not novel ASL generation.
> Retrieved poses must be checked by qualified ASL signers before any real use.
> Never present synthetic poses as ASL.

The notebook:
1. installs libraries
2. loads a small set of real How2Sign 2D pose clips (or clearly-labelled synthetic data for testing)
3. validates sentences and pose files (`[T, 274]`)
4. computes pretrained sentence embeddings (`all-MiniLM-L6-v2`)
5. builds a context-aware retrieval index (text-only vs previous-sentence/pose context)
6. evaluates with an honest train/holdout split
7. saves the index for local inference


In [ ]:
# STEP 1: install (small, CPU-friendly)
!pip install -q sentence-transformers numpy pandas


## STEP 2: configure paths

For **real** data set `USE_SYNTHETIC = False` and point `METADATA_CSV` / `POSE_DIR`
at your How2Sign clips. Build `metadata.csv` locally with:

```
python ml/setup_data.py real --csv how2sign_realigned_train.csv --pose-dir data/poses --output data/metadata.csv
```


In [ ]:
# STEP 2: configuration
import os, json, pickle
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List
import numpy as np

USE_SYNTHETIC = False          # set True ONLY for pipeline testing (NOT real ASL)
METADATA_CSV = "data/metadata.csv"
POSE_DIR = "data/poses"
OUTPUT_INDEX = "models/retrieval_index.pkl"
POSE_DIM = 274

Path("models").mkdir(exist_ok=True)


## STEP 3: dataset loading and validation

In [ ]:
# STEP 3: dataset structures + loaders
@dataclass
class PoseSample:
    sample_id: str
    sentence: str
    video_id: str
    narrative_id: str
    start_time: float
    end_time: float
    pose: Optional[np.ndarray]
    prev_sample_id: Optional[str] = None
    prev_sentence: Optional[str] = None
    is_synthetic: bool = False


def load_real(metadata_csv, pose_dir):
    import csv
    pose_dir = Path(pose_dir)
    id_to_row, rows = {}, []
    with open(metadata_csv, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    id_to_row = {r["sample_id"]: r for r in rows}

    samples, missing, bad = [], 0, 0
    for r in rows:
        sid = r["sample_id"]
        path = pose_dir / f"{sid}.npy"
        if not path.exists():
            path = pose_dir / f"{sid}.npz"
        pose = None
        if path.exists():
            arr = np.load(str(path))
            arr = arr[arr.files[0]] if path.suffix == ".npz" else arr
            if arr.ndim == 3:
                arr = arr.reshape(arr.shape[0], -1)
            if arr.ndim == 2 and arr.shape[1] == POSE_DIM:
                pose = arr.astype(np.float32)
            else:
                bad += 1
        else:
            missing += 1
        prev_id = r.get("prev_sample_id") or None
        samples.append(PoseSample(
            sample_id=sid,
            sentence=r["sentence"],
            video_id=r.get("video_id", ""),
            narrative_id=r.get("narrative_id", ""),
            start_time=float(r.get("start_time", 0) or 0),
            end_time=float(r.get("end_time", 0) or 0),
            pose=pose,
            prev_sample_id=prev_id,
            prev_sentence=id_to_row.get(prev_id, {}).get("sentence") if prev_id else None,
        ))
    print(f"Loaded {len(samples)} samples | missing pose: {missing} | bad shape: {bad}")
    return samples


def make_synthetic(n=30, seed=42):
    rng = np.random.default_rng(seed)
    sentences = [
        "Hello, how are you today?", "My name is Sam.",
        "I am learning American Sign Language.", "Thank you for your patience.",
        "Please show me the sign for water.", "I do not understand.",
        "Can you sign that again?", "What is your name?", "Nice to meet you.",
        "See you tomorrow.", "The weather is nice today.", "I need help with this.",
    ]
    out = []
    for i in range(n):
        frames = int(rng.integers(24, 60))
        t = np.linspace(0, 2 * np.pi, frames)[:, None]
        freq = rng.uniform(0.5, 2.0, (1, POSE_DIM))
        phase = rng.uniform(0, np.pi, (1, POSE_DIM))
        pose = (np.sin(t * freq + phase) * 0.15 + 0.5).astype(np.float32)
        out.append(PoseSample(
            sample_id=f"syn_{i:03d}", sentence=sentences[i % len(sentences)],
            video_id=f"synthetic_video_{i // 5}", narrative_id="synthetic",
            start_time=float(i * 4), end_time=float(i * 4 + 3), pose=pose,
            prev_sample_id=f"syn_{i-1:03d}" if i > 0 else None,
            prev_sentence=sentences[(i - 1) % len(sentences)] if i > 0 else None,
            is_synthetic=True,
        ))
    return out


if USE_SYNTHETIC:
    print("=" * 60)
    print("WARNING: SYNTHETIC data — random sinusoids, NOT real ASL.")
    print("For pipeline testing only. Never present as ASL output.")
    print("=" * 60)
    samples = make_synthetic(30)
else:
    samples = load_real(METADATA_CSV, POSE_DIR)

pose_samples = [s for s in samples if s.pose is not None]
print(f"Samples with pose: {len(pose_samples)}")
if not pose_samples:
    raise SystemExit("No pose data. Run ml/setup_data.py or set USE_SYNTHETIC=True for testing.")


## STEP 4: sentence embeddings and retrieval index

In [ ]:
# STEP 4: embed sentences and build index
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

sentences = [s.sentence for s in pose_samples]
emb = model.encode(sentences, convert_to_numpy=True, show_progress_bar=True)
emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)

def summarize_pose(pose, dim=32):
    if pose is None or len(pose) == 0:
        return np.zeros(dim * 2, dtype=np.float32)
    sub = pose[:, :dim]
    return np.concatenate([sub.mean(axis=0), sub.std(axis=0)]).astype(np.float32)

index = []
for s, e in zip(pose_samples, emb):
    index.append({
        "sample_id": s.sample_id, "sentence": s.sentence,
        "text_emb": e, "pose_feat": summarize_pose(s.pose), "pose": s.pose,
        "has_pose": True,
    })
print(f"Index built: {len(index)} entries")


## STEP 5: context-aware retrieval (fixed weighted similarity)

This is a simple ranking function, not a trained Transformer.

In [ ]:
# STEP 5: retrieval functions
MATCH_THRESHOLD = 0.30
TEXT_WEIGHT = 0.75
POSE_WEIGHT = 0.25

def retrieve(index, query, mode="text_only", prev_sentence=None, prev_pose=None):
    q = model.encode([query], convert_to_numpy=True)[0]
    q = q / (np.linalg.norm(q) + 1e-9)
    ctx = None
    if mode == "context_aware" and prev_sentence:
        ctx = model.encode([f"{prev_sentence} [SEP] {query}"], convert_to_numpy=True)[0]
        ctx = ctx / (np.linalg.norm(ctx) + 1e-9)
    pf = None
    if mode == "context_aware" and prev_pose is not None:
        pf = summarize_pose(prev_pose)
        pf = pf / (np.linalg.norm(pf) + 1e-9)

    best, best_i = -np.inf, None
    for i, e in enumerate(index):
        if mode == "text_only" or ctx is None:
            score = float(np.dot(q, e["text_emb"]))
        else:
            txt = TEXT_WEIGHT * float(np.dot(q, e["text_emb"])) + (1 - TEXT_WEIGHT) * float(np.dot(ctx, e["text_emb"]))
            if pf is not None and e["pose_feat"] is not None:
                p = e["pose_feat"] / (np.linalg.norm(e["pose_feat"]) + 1e-9)
                score = (1 - POSE_WEIGHT) * txt + POSE_WEIGHT * float(np.dot(pf, p))
            else:
                score = txt
        if score > best:
            best, best_i = score, i

    if best_i is None or best < MATCH_THRESHOLD:
        return {"matched_sample_id": None, "score": float(best), "pose": None,
                "reason": f"score {best:.2f} below threshold {MATCH_THRESHOLD:.2f}", "mode": mode}
    e = index[best_i]
    return {"matched_sample_id": e["sample_id"], "matched_sentence": e["sentence"],
            "score": float(best), "pose": e["pose"], "reason": None, "mode": mode}

print("Retrieval functions ready.")


## STEP 6: honest evaluation

- `self_top1`: with the full index (query included), is the top result the query itself? Pipeline sanity only.
- `heldout_mean_score`: index built on a train split; mean top-1 cosine score on held-out samples. A similarity proxy, **not** accuracy.


In [ ]:
# STEP 6: split evaluation
rng = np.random.default_rng(42)
order = rng.permutation(len(pose_samples))
n_test = max(1, int(len(pose_samples) * 0.25))
test_pos = set(int(i) for i in order[:n_test])
train = [s for i, s in enumerate(pose_samples) if i not in test_pos]
test = [s for i, s in enumerate(pose_samples) if i in test_pos]
id_map = {s.sample_id: s for s in pose_samples}

def build(samples_subset):
    sub = [s for s in samples_subset]
    e = model.encode([s.sentence for s in sub], convert_to_numpy=True)
    e = e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-9)
    return [{"sample_id": s.sample_id, "sentence": s.sentence, "text_emb": v,
             "pose_feat": summarize_pose(s.pose), "pose": s.pose, "has_pose": True}
            for s, v in zip(sub, e)]

full_idx = build(pose_samples)
train_idx = build(train)
results = {}
for mode in ["text_only", "context_aware"]:
    hits = 0
    for s in pose_samples:
        pp = id_map[s.prev_sample_id].pose if s.prev_sample_id in id_map else None
        r = retrieve(full_idx, s.sentence, mode, s.prev_sentence, pp)
        hits += int(r["matched_sample_id"] == s.sample_id)
    scores = []
    for s in test:
        pp = id_map[s.prev_sample_id].pose if s.prev_sample_id in id_map else None
        r = retrieve(train_idx, s.sentence, mode, s.prev_sentence, pp)
        scores.append(r["score"])
    results[mode] = {"self_top1": hits / len(pose_samples),
                     "heldout_mean_score": float(np.mean(scores)) if scores else 0.0,
                     "num_train": len(train), "num_test": len(test)}
    print(f"{mode:14s} self_top1={results[mode]['self_top1']:.3f} "
          f"heldout_mean_score={results[mode]['heldout_mean_score']:.3f}")
print("\nNOTE: these are not ASL accuracy scores. Real evaluation needs qualified signers.")


## STEP 7: save and download the index

In [ ]:
# STEP 7: save index in the format ml/inference.py expects
payload = {
    "model_name": MODEL_NAME,
    "is_synthetic": bool(USE_SYNTHETIC),
    "index": [{k: v for k, v in e.items() if k != "pose"} for e in index],
    "index_with_pose": index,
}
Path(OUTPUT_INDEX).parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_INDEX, "wb") as f:
    pickle.dump(payload, f)
with open("models/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved", OUTPUT_INDEX)

try:
    from google.colab import files
    files.download(OUTPUT_INDEX)
    files.download("models/eval_results.json")
except ImportError:
    print("Not in Colab — copy models/retrieval_index.pkl into SignLingo-AI-Mini/models/")


## Next steps

```
# build the index (real data)
python ml/train.py --data-dir data --eval

# or synthetic smoke test (testing only)
python ml/train.py --synthetic --eval

# run the app from SignLingo-AI-Mini/
uvicorn backend.main:app --reload --port 8000
cd frontend && npm run dev
```
